# Feature Engineering: Pricing Diamonds

**Goal:** see how better *inputs* (features) reduce prediction error, using the **same model** every time.

Our dataset has just **10 columns**, and by the end you will understand every one of them.

| Step | Topic | Question it answers |
|---|---|---|
| 0 | Load data | What are we working with? |
| 1 | Scoreboard + baseline | What score do we need to beat? |
| 2 | Bad and missing values | Can we trust every number? |
| 3 | Categorical encoding | How do we use text columns? |
| 4 | Domain features | Can we hand the model what an expert knows? |
| 5 | Transformations | What about skewed and curved data? |
| 6 | Scaling | Why do feature scales matter? |
| 7 | Your turn | Invent your own feature |
| 8 | Recap | What helped, and what didn't? |

> **Rule for this notebook:** change *one thing at a time* and **measure** it.
>
> **Run the cells in order, once.** Each step modifies the table `X` built in the previous step. If something goes wrong, use *Runtime -> Restart and run all*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge

plt.rcParams["figure.figsize"] = (7, 4)

## 0. Load the data

About **54,000 diamonds** sold by an online retailer. We predict the **price** from the diamond's description. (This is the well-known `diamonds` dataset from the R package *ggplot2*, here loaded from the *seaborn-data* repository on GitHub, so it needs internet access.)

### What the columns mean

| Column | Meaning |
|---|---|
| `carat` | Weight of the diamond (1 carat = 0.2 grams) |
| `cut` | Quality of the cut: Fair < Good < Very Good < Premium < Ideal |
| `color` | Color grade from **J** (worst, slightly yellow) to **D** (best, colorless) |
| `clarity` | How free of tiny flaws: I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (flawless) |
| `depth` | Total depth as a percentage of the width (`z / average of x and y`) |
| `table` | Width of the flat top as a percentage of the widest point |
| `x`, `y`, `z` | Length, width and depth in **millimeters** |
| **`price`** | **Price in US dollars: the target we predict** |

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
df = pd.read_csv(url)

y = df["price"]          # the target: what we want to predict
print("Rows, columns:", df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df["price"].hist(bins=50, ax=ax[0]); ax[0].set_title("price ($)")
df["carat"].hist(bins=50, ax=ax[1]); ax[1].set_title("carat")
plt.tight_layout(); plt.show()

## 1. The scoreboard and a baseline

We need a fair way to compare feature sets. The function below is our **scoreboard**:

> You give it a table of features. It tells you how far off its predicted prices are, **on average, in dollars** (this is the *RMSE*). **Lower is better.**

Inside, it uses a prediction model and a testing procedure that we'll cover later in the course. For now treat it as a **black box**. It only *scores*; it never changes your data. So the table you hand it must already be **all numbers with no gaps**.

In [ ]:
results = {}     # the scoreboard remembers every score here

def scoreboard(X, name):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(Ridge(alpha=10), X, y, cv=cv, scoring="neg_root_mean_squared_error", error_score="raise")
    results[name] = -scores.mean()
    print(f"{name:<34} average error = ${-scores.mean():>9,.0f}")

### Baseline: the numeric columns, exactly as they come

In [ ]:
num = df[["carat", "depth", "table", "x", "y", "z"]].copy()    # the six numeric columns

scoreboard(num, "1. Baseline (numbers only)")

That number is our **reference line**. Everything from here on is measured against it.

## 2. Bad and missing values

This table has no blank cells, so are all the numbers trustworthy? **Always check the minimum and maximum of every column.**

In [ ]:
num.describe().round(2)

Look at `x`, `y` and `z` (sizes in millimeters):

- The **minimum is 0**. A diamond with zero length does not exist.
- The **maximum of `y` is 58.9 mm** and of `z` is 31.8 mm, while the typical diamond is about 5 mm wide. Those are typos.

These are **hidden missing values**: nobody left a blank, but the number is wrong. The fix is the same as for blanks: treat the impossible values as *unknown*, then fill them with the **median** (the middle value, which is not thrown off by extremes). We also add a **flag column** so the model knows which rows were repaired.

In [ ]:
sizes = ["x", "y", "z"]
bad = (num[sizes] < 1) | (num[sizes] > 20)     # True where a size is impossible
print("Impossible sizes per column:")
print(bad.sum().to_string())

num[sizes] = num[sizes].mask(bad)              # turn impossible values into gaps (NaN)

X = num.fillna(num.median())                   # fill each gap with its column's median
X["size_was_wrong"] = bad.any(axis=1).astype(int)    # 1 = this row was repaired

scoreboard(X, "2. + fixed bad sizes")

> **Think:** the improvement is small. Why? Only a few dozen rows out of 54,000 were affected. But those rows could be very misleading to the model, and the cost of fixing them is nothing.

## 3. Categorical encoding

`cut`, `color` and `clarity` are text, and so far we ignored them. Models need numbers. All three are **ordinal**: they have a natural order from worst to best, so we map the grades to numbers 1, 2, 3, ... (higher = better).

In [ ]:
cut_grade     = {"Fair": 1, "Good": 2, "Very Good": 3, "Premium": 4, "Ideal": 5}
color_grade   = {"J": 1, "I": 2, "H": 3, "G": 4, "F": 5, "E": 6, "D": 7}              # D is best
clarity_grade = {"I1": 1, "SI2": 2, "SI1": 3, "VS2": 4, "VS1": 5, "VVS2": 6, "VVS1": 7, "IF": 8}

X["cut"]     = df["cut"].map(cut_grade)
X["color"]   = df["color"].map(color_grade)
X["clarity"] = df["clarity"].map(clarity_grade)

scoreboard(X, "3. + grades as numbers")

### Is "equal steps" a fair assumption?

Mapping to 1, 2, 3... says that going from `SI2` to `SI1` is worth **exactly the same** as going from `VVS1` to `IF`. Is it? Probably not.

**One-hot encoding** drops that assumption: it makes one 0/1 column per grade, so the model can learn the value of *each* grade separately. Here is what that looks like for a few rows:

In [ ]:
pd.get_dummies(df["cut"].head(6), dtype=int)       # each row has a 1 in exactly one column

In [ ]:
X = pd.concat([X, pd.get_dummies(df[["cut", "color", "clarity"]], dtype=int)], axis=1)
print("Features now:", X.shape[1])

scoreboard(X, "3b. + one-hot grades")

> **Takeaway:** "ordinal for ordered, one-hot for unordered" is a good first rule, but one-hot is the safer choice when you're unsure the steps are equal. Here we keep *both* versions, because the numeric grades are useful in the next step.

## 4. Domain-derived features

Ask: **"What does a diamond expert know that the model does not?"**

| New feature | Idea |
|---|---|
| `volume` | `x * y * z`: how big the stone really is in 3D |
| Magic sizes | Buyers shop for round numbers. A 1.00 carat stone costs much more than a 0.99 carat stone. Flags for `carat >= 0.5`, `1`, `1.5`, `2` |
| Size x quality | A better stone is worth more **per carat**, so price behaves like *size multiplied by quality*, not size plus quality |

A model like ours can **add** columns, but it cannot **multiply** them by itself. We do the multiplication for it.

In [ ]:
X["volume"] = X["x"] * X["y"] * X["z"]

scoreboard(X, "4a. + volume")

In [ ]:
for size in [0.5, 1, 1.5, 2]:
    X[f"at_least_{size}_carat"] = (X["carat"] >= size).astype(int)

scoreboard(X, "4b. + magic carat sizes")

In [ ]:
X["carat_x_cut"]     = X["carat"] * X["cut"]
X["carat_x_color"]   = X["carat"] * X["color"]
X["carat_x_clarity"] = X["carat"] * X["clarity"]

scoreboard(X, "4c. + size x quality")

Look at that last jump: this is domain knowledge paying off.

> **Think:** why not just add `cut + color + clarity` into one "quality score"? Try it if you like. Adding columns gives the model nothing new, because it can already add. **Only products (and other non-linear combinations) give it something it couldn't do before.**

## 5. Transformations: skew and curves

Two typical problems:

1. **Skew**: most diamonds are small, with a long tail of big ones.
2. **Curves**: price does not grow in a straight line with carat. It accelerates.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))

X["carat"].hist(bins=50, ax=ax[0]);            ax[0].set_title("carat (skewed)")
np.log1p(X["carat"]).hist(bins=50, ax=ax[1]);  ax[1].set_title("carat after log")

sample = df.sample(3000, random_state=0)       # 3,000 dots are enough to see the shape
ax[2].scatter(sample["carat"], sample["price"], s=4, alpha=0.4)
ax[2].set_title("price vs. carat (a curve, not a line)"); ax[2].set_xlabel("carat")

plt.tight_layout(); plt.show()

- **Log** (`log1p(x) = log(1 + x)`) squeezes a long tail, so that going from 0.3 to 0.6 carat counts about as much as going from 1.5 to 3 carat.
- **Squaring** lets a straight-line model bend upward.

We add one of each:

In [ ]:
X["carat_squared"] = X["carat"] ** 2           # lets the model bend upward
X["log_volume"]    = np.log1p(X["volume"])     # squeezes the long tail

scoreboard(X, "5. + curve and log")

> **Think:** the gain here is small. That is normal, and it is *why we measure*. Most of the curve was already captured by "size x quality". Not every feature earns its place; a measured "meh" is a useful result.

## 6. Scaling

Features live on very different scales:

In [ ]:
X[["carat", "depth", "volume", "carat_x_clarity"]].agg(["mean", "std"]).round(1)

`depth` is around 62 while `carat` is around 0.8. Many models, including the one inside our scoreboard, **penalize large weights**. A feature with big numbers needs only a tiny weight, so it is barely penalized, while a feature with small numbers needs a large weight and is penalized heavily. The units alone decide who gets treated fairly.

**Fix: standardize.** Subtract the column's mean and divide by its standard deviation, so every column has mean 0 and spread 1:

$$z = \frac{x - \text{mean}}{\text{std}}$$

We do this **last**, after all the columns are in their final form.

In [ ]:
X = (X - X.mean()) / X.std()

scoreboard(X, "6. + scaling")

> **Think:** you likely see almost no change. Why? With **54,000 diamonds**, the penalty is tiny compared to the total error, so unfair treatment barely matters. Scaling matters much more with **small datasets** and with models such as k-nearest-neighbors or neural networks. It is cheap and safe, so it's still standard practice.

## 7. Your turn (about 8 minutes)

Invent **one** new feature and test whether it improves the score. Write it in the marked line using columns from `df` (the original, unscaled data). The next lines tidy it up and add it to a copy of the final table.

**Idea bank**

| Idea | Code |
|---|---|
| How round is the stone? | `df["x"] / df["y"]` |
| Height vs. width | `df["z"] / df["x"]` |
| Top area times depth | `df["table"] * df["depth"]` |
| Colorless flag (D, E, F) | `df["color"].isin(["D", "E", "F"])` |
| Near-flawless flag | `df["clarity"].isin(["IF", "VVS1", "VVS2"])` |
| Big and flawless | `df["carat"] * df["clarity"].isin(["IF", "VVS1", "VVS2"])` |

**Rules**
- Use only *input* columns. **Never build a feature from `price`** (that would be cheating: the model would be handed the answer).
- A feature that doesn't help is a valid, useful result. Report it!

In [ ]:
X_mine = X.copy()

# ---- YOUR IDEA HERE: one line that produces a new column -------------
new = df["x"] / df["y"]
# -----------------------------------------------------------------------

new = new.astype(float)                                    # True/False -> 1.0/0.0
new = new.replace([np.inf, -np.inf], np.nan)               # division by zero -> gap
new = new.fillna(new.median())                             # fill gaps (as in step 2)
X_mine["MyFeature"] = (new - new.mean()) / new.std()       # scale it (as in step 6)

scoreboard(X_mine, "7. + my feature")
print("Change vs. step 6: $", round(results["7. + my feature"] - results["6. + scaling"]), "(negative = better)")

## 8. Recap

### The scoreboard

In [ ]:
board = pd.DataFrame({"average error ($)": pd.Series(results).round(0)})
first = board["average error ($)"].iloc[0]
board["change vs. baseline"] = ((board["average error ($)"] / first - 1) * 100).round(1).astype(str) + " %"
print(board.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
labels = [k.split(". ", 1)[1] if ". " in k else k for k in results]
ax.barh(labels[::-1], list(results.values())[::-1], color="#2a9d8f")
ax.set_xlabel("Average prediction error ($), lower is better")
ax.set_title("Same model, better features")
plt.tight_layout(); plt.show()

### Which steps paid off?

- **Big wins:** using the text columns (step 3) and **domain knowledge** (step 4, especially size x quality).
- **Small wins:** repairing bad values, transforms, scaling. Cheap, safe, but not dramatic *on this dataset and this model*.

### Take-home messages

1. **Look at your data first.** Zeros and absurd maximums are hidden missing values.
2. **Domain knowledge beats generic tricks.** "Price = size x quality" did more than any transformation.
3. **Measure every change.** Some features earn their place and some don't.

### Where to go next
- Training and testing properly (the scoreboard's black box, opened)
- Tree-based models, which can find "size x quality" interactions on their own. What would remain worth engineering?
- Try the same workflow on another dataset, such as bike-share demand (dates and weather)